# 累计电量差分
每天一行，96 个 15 分钟累计电量读数。倍率按原始计量口径设置。

In [ ]:
from load_common import export, hourly_frame, fill_meter_gaps, PERIODS
import numpy as np
import pandas as pd

YEAR = 2026
TARIFF = "inputs/tariff.xlsx"
INPUT = "inputs/cumulative_meter.xlsx"
OUTPUT = "outputs/cumulative_meter_8760.xlsx"
SHEET = 0
DATE_COL = "数据日期"
MULTIPLIER = 1.0  # 二次侧电量填综合倍率，已还原填 1


In [ ]:
raw = pd.read_excel(INPUT, sheet_name=SHEET)
slots = pd.timedelta_range("0min", periods=96, freq="15min")
slot_columns = {f"{slot.components.hours}:{slot.components.minutes:02d}": slot for slot in slots}
normalized = {str(col).strip().lstrip("0") or "0": col for col in raw.columns}
columns = [normalized[label.lstrip("0")] for label in slot_columns]
dates = pd.to_datetime(raw[DATE_COL], errors="raise").dt.normalize()
if dates.isna().any() or dates.duplicated().any():
    raise ValueError("日期为空或有重复，请先筛选单只表并确认重复行")
timestamps = (dates.to_numpy()[:, None] + slots.to_numpy()).ravel()
readings = raw[columns].apply(pd.to_numeric, errors="raise").to_numpy().ravel()
meter = pd.Series(readings, index=pd.DatetimeIndex(timestamps)).sort_index().asfreq("15min")
energy = (meter.shift(-1) - meter) * MULTIPLIER
print("负增量区间数：", (energy < 0).sum())
# 按手册将换表或校表造成的负增量裁到 0。完整四个区间才计入实测小时。
hourly = energy.clip(lower=0).resample("h").sum(min_count=4)

frame = hourly_frame(YEAR, TARIFF)
frame["负载"], frame["负载来源"] = fill_meter_gaps(hourly, frame.index)
audit = frame.groupby(["月份", "负载来源"], as_index=False).agg(
    小时数=("负载", "size"), 电量_kWh=("负载", "sum")
)
export(frame, OUTPUT, audit)
